In [1]:
import os

os.environ["JAX_PLATFORM_NAME"] = "cpu"

import joblib
import optax.projections
%load_ext autoreload
%autoreload 2

from collections import namedtuple
from scipy.stats import binned_statistic
import jax
import jax.numpy as jnp
import equinox as eqx

import numpy as np
import matplotlib.pyplot as plt

from qdots_qll.distributions import Distribution, update_particles_locations, update_weights

import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error

from jax import jit
from jax.scipy.linalg import expm

from qdots_qll.models.single_dot_weak_coupling_GAME import *

from qdots_qll.resamplers import LWResamplerBounds

from qdots_qll.exp_design import OptimizeInitialStateMeasurements, MaxDetFimExpDesign

from tensorflow_probability.substrates import jax as tfp
import optax

import jax.tree_util as jtu


def tree_stack(trees):
    return jax.tree.map(lambda *v: jnp.stack(v), *trees)


def tree_unstack(tree):
    leaves, treedef = jax.tree.flatten(tree)
    return [treedef.unflatten(leaf) for leaf in zip(*leaves, strict=True)]


def transpose_results(pytree):
    return tree_stack(list(map(list, zip(*tree_unstack(tree_unstack(pytree))))))


In [2]:


def process_data(data_array, no_bins=300):
    ProcessedData = namedtuple('ProcessedData', [
        'outcomes', 'times', 'exp_values', 'covs',
        'p_init_states', 'p_basis', 'cumtimes_binned',
        'covs_mean_binned', 'covs_std_binned',
        'det_covs_mean_binned', 'det_covs_std_binned'
    ])

    # data_array = re
    # no_bins = 300

    outcomes, times, exp_values, covs, p_init_states, p_basis = data_array

    outcomes = outcomes.transpose(1, 0, 2)
    times = times.T
    exp_values = exp_values.transpose(1, 0, 2)
    covs = covs.transpose(1, 0, 2, 3)
    p_init_states = p_init_states.transpose(1, 0, 2)
    p_basis = p_basis.transpose(1, 0, 2)

    no_rv = covs.shape[2]
    no_runs = covs.shape[0]
    no_iterations = covs.shape[1]

    covs_mean = np.mean(covs, axis=0)
    covs_std = np.std(covs, axis=0)

    det_covs_runs = jax.vmap(lambda run: jax.vmap(lambda run: jnp.linalg.det(run))(run))(covs)

    cumtimes = np.cumsum(times, axis=1)

    cumtimes_binned, _, _ = binned_statistic(cumtimes.flatten(), cumtimes.flatten(), statistic='mean', bins=no_bins)

    det_covs_mean_binned, _, _ = binned_statistic(cumtimes.flatten(), det_covs_runs.flatten(), statistic='mean',
                                                  bins=no_bins)

    det_covs_std_binned, _, _ = binned_statistic(cumtimes.flatten(), det_covs_runs.flatten(), statistic='std',
                                                 bins=no_bins)

    covs_mean_binned = np.zeros([cumtimes_binned.shape[0], no_rv, no_rv])
    covs_std_binned = np.zeros([cumtimes_binned.shape[0], no_rv, no_rv])

    for i in range(no_rv):
        for j in range(no_rv):
            y, _, _ = binned_statistic(cumtimes.flatten(), covs.reshape(-1, *covs.shape[2:])[:, i, j], statistic='mean',
                                       bins=no_bins)
            covs_mean_binned[:, i, j] = y

            y, _, _ = binned_statistic(cumtimes.flatten(), covs.reshape(-1, *covs.shape[2:])[:, i, j], statistic='std',
                                       bins=no_bins)
            covs_std_binned[:, i, j] = y

    # return outcomes, times, exp_values, covs, p_init_states, p_basis, cumtimes_binned, covs_mean_binned, covs_std_binned, det_covs_mean_binned, det_covs_std_binned
    processed_data = ProcessedData(
        outcomes, times, exp_values, covs,
        p_init_states, p_basis, cumtimes_binned,
        covs_mean_binned, covs_std_binned,
        det_covs_mean_binned, det_covs_std_binned
    )

    return processed_data


In [406]:

list_of_names = []
for f_name in os.listdir('results_one_qubit/'):
    if f_name.startswith("run_2024-06-06"):
        list_of_names.append(str(f_name))
        # print(f_name)
        # data_list.append(joblib.load('results_one_qubit/' + str(f_name)))

In [407]:
data_list = []
for i_str in sorted(list_of_names):
    data_list.append(joblib.load('results_one_qubit/' + i_str))
    
data_list.append(joblib.load('results_one_qubit/run_2024-06-10_16-08-40'))
    

In [408]:
sorted(list_of_names)

In [409]:

list_of_data = [process_data(i) for i in data_list]
names_list = ["full adaptive", 't opt', 'trace', 'vanilla', 'adapt correct']

In [423]:
fig, axs = plt.subplots(2, 2, figsize=(8, 8), dpi=300)

for i, ax in enumerate(axs.flatten()):
    for j, data_i in enumerate(list_of_data):
        y = data_i.covs_mean_binned[:, i, i]
        y_std = data_i.covs_std_binned[:, i, i]
        x = data_i.cumtimes_binned

        ax.plot(x, y, label=names_list[j])
        ax.fill_between(x=x, y1=y + y_std, y2=y - y_std, alpha=0.2)
        ax.loglog()

plt.legend()
plt.show()

fig, ax = plt.subplots(figsize=(5, 5), dpi=300)
for j, data_i in enumerate(list_of_data):
    ax.hist(np.array(data_i.times).flatten(), bins=200, label=names_list[j], alpha=0.5)

plt.legend()
plt.show()

fig, ax = plt.subplots(figsize=(4, 4), dpi=300)

for j, data_i in enumerate(list_of_data):
    x = data_i.cumtimes_binned
    y = data_i.det_covs_mean_binned[:]
    y_std = data_i.det_covs_std_binned[:]

    ax.plot(x, y, label=names_list[j])
    # ax.fill_between(x=x, y1=y + y_std, y2=y - y_std, alpha=0.3)
    ax.loglog()

plt.legend()
plt.show()








In [412]:
fig, axs = plt.subplots(2, 3, )

# states_names = ["1", "2", "3", "4"]
for i, ax in enumerate(axs.flatten()):
    data_i = list_of_data[i]

    x = np.arange(data_i.p_init_states.shape[1])
    y = data_i.p_init_states.mean(axis=0)[:, :]

    ax.set_title(names_list[i])
    ax.plot(x, y, )  #label=names_list[i])
    ax.set_ylim(0, 1)

# plt.legend()
plt.tight_layout()
plt.show()

fig, axs = plt.subplots(2, 2, )

# states_names = ["1", "2", "3", "4"]
for i, ax in enumerate(axs.flatten()):
    data_i = list_of_data[i]
    x = np.arange(data_i.p_basis.shape[1])
    y = data_i.p_basis.mean(axis=0)[:, :]
    # y_std = data_i.p_init_states.std(axis=0)[:, i]

    ax.set_title(names_list[i])
    ax.plot(x, y, )  #label=names_list[i])
    ax.axhline(0.333, alpha=0.4)
    # ax.fill_between(x=x, y1=y + y_std, y2=y - y_std, alpha=0.3)
    ax.set_ylim(0, 1)

# plt.legend()
plt.tight_layout()
plt.show()

In [9]:
m = SingleDotWeakCouplingGAME()

In [10]:
true_parameters

# FIM with flat distribution


In [413]:
names_list

In [419]:
times = jnp.linspace(0, 100., 500)


flat_p_state =  jnp.ones(4) / 4
flat_p_basis = jnp.ones(3)/3

pointer_data = 4
adaptive_p_state = list_of_data[pointer_data].p_init_states.mean(0)[-1]
adaptive_p_basis = list_of_data[pointer_data].p_basis.mean(0)[-1]

expval_adaptive = np.mean(list_of_data[pointer_data].exp_values, axis=0)[-1]

In [420]:
list_of_data[0].times.flatten()

In [424]:


fim_times_flat = jax.vmap(lambda t: m.fim(true_parameters, t, flat_p_state, flat_p_basis))(times)

det_fim_times_flat = jax.vmap(lambda mat: jnp.linalg.det(mat))(fim_times_flat)


fig, axs = plt.subplots(2, 2)

fim_times_adapt = jax.vmap(lambda t: m.fim(true_parameters, t, adaptive_p_state, adaptive_p_basis))(times)
det_fim_times_adapt = jax.vmap(lambda mat: jnp.linalg.det(mat))(fim_times_adapt)

for i, ax in enumerate(axs.flatten()):
    ax.plot(times, fim_times_flat[:, i, i], label="Vanilla")
    ax.plot(times, fim_times_adapt[:, i, i], label="Adaptive")

ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots()

ax.plot(times, det_fim_times_flat, label='vanilla')
ax.plot(times, det_fim_times_adapt, label='adaptive')

ax.hist(np.array(list_of_data[pointer_data].times).flatten(), density=True, label='Adaptive', bins=100, alpha=0.5)

ax.hist(np.array(list_of_data[1].times).flatten(), density=True, label='Vanilla', bins=100, alpha=0.5)

ax.set_title("det F vanilla vs adapt with true pars")
ax.legend()
plt.show()


In [422]:
fim_times_flat = jax.vmap(lambda t: m.fim(expval_adaptive, t, flat_p_state, flat_p_basis))(times)

det_fim_times_flat = jax.vmap(lambda mat: jnp.linalg.det(mat))(fim_times_flat)


fig, axs = plt.subplots(2, 2)

fim_times_adapt = jax.vmap(lambda t: m.fim(expval_adaptive, t, adaptive_p_state, adaptive_p_basis))(times)
det_fim_times_adapt = jax.vmap(lambda mat: jnp.linalg.det(mat))(fim_times_adapt)

for i, ax in enumerate(axs.flatten()):
    ax.plot(times, fim_times_flat[:, i, i], label="Vanilla")
    ax.plot(times, fim_times_adapt[:, i, i], label="Adaptive")
    ax.get_legend()

plt.tight_layout()
plt.show()

fig, ax = plt.subplots()

ax.plot(times, det_fim_times_flat, label='vanilla')
ax.plot(times, det_fim_times_adapt, label='adaptive')

ax.set_title("det F vanilla vs adapt with exp val pars")
ax.legend()
plt.show()


In [25]:
list_of_data[0]._fields

In [13]:


for i in range(4):
    plt.plot(times, fim_times[:, i, i], label=str(i))

plt.legend()
plt.show()

plt.plot(times, det_fim_times)

# Why the final probability distribution does not yield a higher determinant of the fisher information

Let's start with a flat probability distribution, and iterate with our optimizer using the true parameters as particle

In [286]:
from qdots_qll.exp_design import OptimizeInitialStateMeasurementsNoProjection

In [397]:
popt = OptimizeInitialStateMeasurements(iter=5, lr=10)
popt_no_proj = OptimizeInitialStateMeasurementsNoProjection(iter=5, lr=0.1)


In [398]:
key = jax.random.PRNGKey(seed=3)

In [399]:
new_prob_initial_state = flat_p_state
new_prob_measurement_basis = flat_p_basis

p_state_list = []
p_basis_list = []
dettimeslist = []

In [400]:

for i in range(1000):
    key, subkey = jax.random.split(key)
    t = jax.random.uniform(key=subkey, minval=5., maxval=60.0)
    # times_list.append(t)

    new_prob_initial_state, new_prob_measurement_basis = eqx.filter_jit(
        popt.optimize_probability_distribution
    )(
        dist_initial_state=new_prob_initial_state,
        dist_measurement_basis=new_prob_measurement_basis,
        model=m,
        t=t,
        particle=true_parameters,
    )
    if i % 50 == 0:
        dettimes = jax.vmap(
            lambda t: 
                jnp.linalg.det(
                    m.fim(
                        expval_adaptive,
                        t,
                        new_prob_initial_state,
                        new_prob_measurement_basis,
                    )
                )
            
        )(times)
        dettimeslist.append(jnp.max(dettimes[2:]))
        p_state_list.append(new_prob_initial_state)
        p_basis_list.append(new_prob_measurement_basis)


# print(new_prob_initial_state.round(3))
# print(new_prob_measurement_basis.round(3))

In [401]:
plt.plot(jnp.arange(len(p_basis_list))*50, np.array(p_basis_list)[:], '--')

plt.plot(jnp.arange(len(p_basis_list))*50, np.array(dettimeslist)[:], '-', label="det")
plt.legend()
plt.show()


plt.plot(jnp.arange(len(p_basis_list))*50, np.array(p_state_list)[:], '--')

plt.plot(jnp.arange(len(p_basis_list))*50, np.array(dettimeslist)[:], '-', label="det")
plt.legend()
plt.show()

In [382]:
plt.plot(dettimeslist)

In [381]:
fim_times_flat = jax.vmap(lambda t: m.fim(expval_adaptive, t, flat_p_state, flat_p_basis))(times)

det_fim_times_flat = jax.vmap(lambda mat: jnp.linalg.det(mat))(fim_times_flat)


fig, axs = plt.subplots(2, 2)

fim_times_adapt = jax.vmap(lambda t: m.fim(expval_adaptive, t, new_prob_initial_state, new_prob_measurement_basis))(times)
det_fim_times_adapt = jax.vmap(lambda mat: jnp.linalg.det(mat))(fim_times_adapt)

for i, ax in enumerate(axs.flatten()):
    ax.plot(times, fim_times_flat[:, i, i], label="Vanilla")
    ax.plot(times, fim_times_adapt[:, i, i], label="Adaptive")
    ax.get_legend()

plt.tight_layout()
plt.show()

fig, ax = plt.subplots()

ax.plot(times, det_fim_times_flat, label='vanilla')
ax.plot(times, det_fim_times_adapt, label='adaptive')

ax.set_title("det F vanilla vs adapt with exp val pars")
ax.legend()
plt.show()

In [48]:
all_accumulate_with_adam.p_init_states.mean(0)[-1].round(3)

In [51]:
times = jnp.linspace(0, 100., 500)

pstate = all_accumulate_with_adam.p_init_states.mean(0)[-1]
pbasis = all_accumulate_with_adam.p_basis.mean(0)[-1]
fim_times = jax.vmap(lambda t: m.fim(true_parameters, t, pstate, pbasis))(times)

det_fim_times = jax.vmap(lambda t: jnp.linalg.det(m.fim(true_parameters, t, pstate, pbasis)))(times)

for i in range(4):
    plt.plot(times, fim_times[:, i, i], label=str(i))

plt.legend()
plt.show()

plt.plot(times, det_fim_times)
plt.show()

In [52]:
0.008 / 0.005